# 🎙️ VoiceLab 2.0 — Chatterbox GPU Backend Server

This Colab notebook hosts the **Chatterbox TTS** voice-cloning model on a free GPU and exposes a secure HTTPS API for your **VoiceLab** web interface.

### Quick Setup Instructions:
1. Make sure your runtime is set to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).
2. Run all cells in order (`Runtime > Run all`).
3. Copy the generated **VOICELAB_API_URL** and **VOICELAB_API_TOKEN** into your VoiceLab sidebar or `.streamlit/secrets.toml`.
4. Keep this Colab tab open while synthesizing speech in VoiceLab.

In [ ]:
# 1. Verify GPU Hardware
import torch

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Please enable GPU: Runtime > Change runtime type > Hardware accelerator > T4 GPU")

In [ ]:
# 2. Install Tested Chatterbox & API Server Dependencies
!pip uninstall -y torchvision
!pip install --no-deps torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q chatterbox-tts==0.1.7 fastapi uvicorn python-multipart pyngrok soundfile

In [ ]:
# 3. Load the Chatterbox Model onto GPU
import torch
import torchaudio
from chatterbox.tts import ChatterboxTTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Chatterbox on {device}...")
model = ChatterboxTTS.from_pretrained(device=device)
print("✓ Chatterbox model loaded successfully! Native sample rate:", model.sr)

In [ ]:
# 4. Configure API Token
import secrets
import os

# Try to load from Colab Secrets, or generate a secure session token
try:
    from google.colab import userdata
    VOICELAB_API_TOKEN = userdata.get("VOICELAB_API_TOKEN")
except Exception:
    VOICELAB_API_TOKEN = None

if not VOICELAB_API_TOKEN:
    # Auto-generate a secure token for this session
    VOICELAB_API_TOKEN = secrets.token_hex(20)

print("✓ Authentication Token active.")

In [ ]:
# 5. Define FastAPI Endpoints (/health and /generate)
import io
import tempfile
import threading
from fastapi import FastAPI, File, Form, UploadFile, Header, HTTPException
from fastapi.responses import Response
import uvicorn

api = FastAPI(title="VoiceLab Chatterbox Backend", docs_url=None, redoc_url=None)
generation_lock = threading.Lock()

@api.get("/health")
def health():
    return {
        "ready": True,
        "model": "chatterbox-tts",
        "sample_rate": model.sr,
        "device": device
    }

@api.post("/generate")
async def generate(
    reference: UploadFile = File(...),
    text: str = Form(...),
    consent: bool = Form(True),
    authorization: str = Header(default=""),
):
    # Token check
    expected_header = f"Bearer {VOICELAB_API_TOKEN}"
    if not secrets.compare_digest(authorization, expected_header):
        raise HTTPException(status_code=401, detail="Unauthorized API Token")
    
    if not (1 <= len(text.strip()) <= 500):
        raise HTTPException(status_code=400, detail="Text must be between 1 and 500 characters")
    
    data = await reference.read(15 * 1024 * 1024)
    if len(data) == 0:
        raise HTTPException(status_code=400, detail="Uploaded reference audio was empty")
    
    if not generation_lock.acquire(blocking=False):
        raise HTTPException(status_code=429, detail="GPU is currently synthesizing another request. Please retry in a few seconds.")
    
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
            f.write(data)
            temp_path = f.name
        
        # Synthesize using Chatterbox
        print(f"Synthesizing text: '{text.strip()[:40]}...' using prompt {temp_path}")
        wav = model.generate(text.strip(), audio_prompt_path=temp_path)
        
        buf = io.BytesIO()
        torchaudio.save(buf, wav.cpu(), model.sr, format="wav")
        return Response(content=buf.getvalue(), media_type="audio/wav")
    except Exception as e:
        print(f"Synthesis error: {e}")
        raise HTTPException(status_code=500, detail=f"Synthesis failed: {str(e)}")
    finally:
        if temp_path and os.path.exists(temp_path):
            try:
                os.remove(temp_path)
            except Exception:
                pass
        generation_lock.release()

print("✓ FastAPI application defined.")

In [ ]:
# 6. Launch Server & Start Public HTTPS Tunnel
import time
import subprocess
import re

# Start Uvicorn in a background thread
server_config = uvicorn.Config(api, host="127.0.0.1", port=8765, log_level="warning")
server = uvicorn.Server(server_config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

for _ in range(40):
    if server.started:
        break
    time.sleep(0.1)

# Check for ngrok token first
public_url = None
try:
    from google.colab import userdata
    NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    NGROK_AUTHTOKEN = os.getenv("NGROK_AUTHTOKEN", "")

if NGROK_AUTHTOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    tunnel = ngrok.connect(8765, "http")
    public_url = tunnel.public_url
    print("Tunneling via pyngrok:", public_url)
else:
    # Zero-signup Free Cloudflare Tunnel
    print("Starting Free Cloudflare HTTPS tunnel (no account needed)...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared-linux-amd64
    cf_proc = subprocess.Popen(
        ["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8765"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    time.sleep(4)
    for _ in range(25):
        line = cf_proc.stderr.readline()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break
        time.sleep(0.3)

print("\n============================================================")
print("🎉 VOICELAB CHATTERBOX BACKEND IS LIVE!")
print("============================================================\n")
print("Copy these credentials into your VoiceLab app:")
print("VOICELAB_API_URL   =", public_url)
print("VOICELAB_API_TOKEN =", VOICELAB_API_TOKEN)
print("\nKeep this Colab tab running while using VoiceLab Voice Studio.")
print("============================================================")